In [2]:
import pandas as pd
import numpy as np
import xlsxwriter as xl
import io
from xlsxwriter.utility import xl_col_to_name

In [5]:
pd.options.display.float_format = '{:.2f}'.format

In [6]:
sourceData = pd.read_excel(
    "C:/Auditor Valuation Framework/exccute.py/Profitability results from FY 2024 to FY 2025.xlsx",
    sheet_name="Detailed"
)
sourceData

,Business ID,Business Title,Type of Business,Underwriting Year,Cedent Name,Cedent Country,Reporting Unit1 Leaf Name,Broker Name,Insured Name,Insured Period Start,...,Detail Amount (Original),Detail Amount (Base),Retroceded Amt Base,Retained Amt Base,Main Class of Business,Class of Business,Registered Date,Closed Date,WS Registered By,WS Closed By
0,NPF1003A,CHEVRON NIGERIA LTD/ESCRAVOS GTL,Non-Prop Facultative,2023,NEM Insurance PLC Nigeria,Nigeria,Nigeria Office,Gallagher Reinsurance Brokers,CHEVRON NIGERIA LTD/ESCRAVOS GTL,2023-01-02,...,1751.75,1751.75,0.00,1751.75,Oil&Gas,All Risk Oil&Gas,2024-05-08 06:24:35,2024-05-08 09:20:25,Frances Nwogbe,Michael Adeyinka
1,NPF1003A,CHEVRON NIGERIA LTD/ESCRAVOS GTL,Non-Prop Facultative,2023,NEM Insurance PLC Nigeria,Nigeria,Nigeria Office,Gallagher Reinsurance Brokers,CHEVRON NIGERIA LTD/ESCRAVOS GTL,2023-01-02,...,-175.18,-175.18,0.00,-175.18,Oil&Gas,All Risk Oil&Gas,2024-05-08 06:24:35,2024-05-08 09:20:25,Frances Nwogbe,Michael Adeyinka
2,NPF1003A,CHEVRON NIGERIA LTD/ESCRAVOS GTL,Non-Prop Facultative,2023,NEM Insurance PLC Nigeria,Nigeria,Nigeria Office,Gallagher Reinsurance Brokers,CHEVRON NIGERIA LTD/ESCRAVOS GTL,2023-01-02,...,-262.76,-262.76,0.00,-262.76,Oil&Gas,All Risk Oil&Gas,2024-05-08 06:24:35,2024-05-08 09:20:25,Frances Nwogbe,Michael Adeyinka
3,NPF1003A,CHEVRON NIGERIA LTD/ESCRAVOS GTL,Non-Prop Facultative,2023,NEM Insurance PLC Nigeria,Nigeria,Nigeria Office,Gallagher Reinsurance Brokers,CHEVRON NIGERIA LTD/ESCRAVOS GTL,2023-01-02,...,623.27,623.27,0.00,623.27,Oil&Gas,All Risk Oil&Gas,2024-07-29 08:04:11,2024-07-29 08:05:30,Frances Nwogbe,Sandra Didiya Yakusak
4,NPF1003A,CHEVRON NIGERIA LTD/ESCRAVOS GTL,Non-Prop Facultative,2023,NEM Insurance PLC Nigeria,Nigeria,Nigeria Office,Gallagher Reinsurance Brokers,CHEVRON NIGERIA LTD/ESCRAVOS GTL,2023-01-02,...,-93.49,-93.49,0.00,-93.49,Oil&Gas,All Risk Oil&Gas,2024-07-29 08:04:11,2024-07-29 08:05:30,Frances Nwogbe,Sandra Didiya Yakusak
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246254,PPT6391A,PEOPLE & PARTNERS PROPERTY AUTO FAC - BANYAN E...,Proportional Treaty,2025,People and Partners Insurance Plc,Cambodia,Asia,NaN,NaN,2025-10-01,...,-280.38,-280.38,0.00,-280.38,Property,Fire (unspecified),2026-01-02 15:41:19,2026-01-02 15:51:26,Alusine Kamara,Alpha Koroma
246255,PPT6391A,PEOPLE & PARTNERS PROPERTY AUTO FAC - BANYAN E...,Proportional Treaty,2025,People and Partners Insurance Plc,Cambodia,Asia,NaN,NaN,2025-10-01,...,-62.94,-62.94,0.00,-62.94,Property,Fire (unspecified),2026-01-02 15:41:19,2026-01-02 15:51:26,Alusine Kamara,Alpha Koroma
246256,PPT6391A,PEOPLE & PARTNERS PROPERTY AUTO FAC - BANYAN E...,Proportional Treaty,2025,People and Partners Insurance Plc,Cambodia,Asia,NaN,NaN,2025-10-01,...,164.46,164.46,0.00,164.46,Property,Fire (unspecified),2026-01-02 15:49:08,2026-01-02 15:52:22,Alusine Kamara,Alpha Koroma
246257,PPT6391A,PEOPLE & PARTNERS PROPERTY AUTO FAC - BANYAN E...,Proportional Treaty,2025,People and Partners Insurance Plc,Cambodia,Asia,NaN,NaN,2025-10-01,...,-40.29,-40.29,0.00,-40.29,Property,Fire (unspecified),2026-01-02 15:49:08,2026-01-02 15:52:22,Alusine Kamara,Alpha Koroma


In [6]:
categoryHeader = "Reporting Unit1 Leaf Name"
codeHeader = "Entry Code"
dataFieldHeader = "Retained Amt Base"
grossDataFieldHeader = "Detail Amount (Base)"
netManagementExpenseRatio = 0.12
sheetName = "Detailed"  # Parameter for Excel sheet name

premiumCode = np.array([100,500])
claimCode = np.array([300,600,900])
commissionCode = np.array([200])

# Load data
sourceData = pd.read_excel(
    "C:/Auditor Valuation Framework/exccute.py/Profitability results from FY 2024 to FY 2025.xlsx",
    sheet_name=sheetName
)

filterItems = pd.unique(sourceData[categoryHeader].dropna())
sourceData[codeHeader] = sourceData[codeHeader].astype(str).str.strip()
sourceData[codeHeader] = pd.to_numeric(sourceData[codeHeader], errors="coerce")

resultTableDictionary = {
    categoryHeader:np.append(np.array(filterItems), "Total"),
    "Net Premium": np.array([]),
    "Net Incurred Claim": np.array([]),
    "Net Commission": np.array([]),
    "Net Technical Margin": np.array([]),
    "Loss Ratio": np.array([]),
    "Commission Ratio": np.array([]),
    "Net Management Expense Ratio": np.array([]),
    "Net Technical Margin Ratio": np.array([]),
    "Net Retro Expense Ratio": np.array([]),
    "Combined Ratio": np.array([])
}

grossPremiumSum = 0
grossPremiumSumArr = np.array([])
premiumTable = pd.DataFrame(columns=sourceData.columns)
claimTable = pd.DataFrame(columns=sourceData.columns)
commissionTable = pd.DataFrame(columns=sourceData.columns)

def ratioAnalysis():

    global grossPremiumSum, grossPremiumSumArr, premiumTable, claimTable, commissionTable

    for item in filterItems:
        print(item)
        filteredTable = sourceData[sourceData[categoryHeader] == item]
        
        premiumSum = 0
        grossPremiumSum = 0
        for code in premiumCode:
            premiumFilteredTable = filteredTable[(filteredTable[codeHeader] >= code) & (filteredTable[codeHeader] <= (code + 99))]
            premiumTable = pd.concat([premiumTable, premiumFilteredTable], ignore_index=True)
            premiumSum += np.sum(premiumFilteredTable[dataFieldHeader])
            grossPremiumSum += np.sum(premiumFilteredTable[grossDataFieldHeader])

        claimSum = 0
        for code in claimCode:
            claimFilteredTable = filteredTable[(filteredTable[codeHeader] >= code) & (filteredTable[codeHeader] <= (code + 99))]
            claimTable = pd.concat([claimTable,claimFilteredTable], ignore_index=True)
            claimSum += np.sum(claimFilteredTable[dataFieldHeader])

        commissionSum = 0
        for code in commissionCode:
            commissionFilteredTable = filteredTable[(filteredTable[codeHeader] >= code) & (filteredTable[codeHeader] <= (code + 99))]
            commissionTable = pd.concat([commissionTable, commissionFilteredTable], ignore_index=True)
            commissionSum += np.sum(commissionFilteredTable[dataFieldHeader])

        netTechnicalMargin = premiumSum + claimSum + commissionSum
        lossRatio = abs(claimSum / premiumSum) if premiumSum != 0 else 0
        commissionRatio = abs(commissionSum / premiumSum) if premiumSum != 0 else 0
        netManagementExpenseRatioLocal = netManagementExpenseRatio
        netTechnicalMarginRatio = abs(netTechnicalMargin/premiumSum) if premiumSum != 0 else 0
        netRetroExpenseRatio = 1 - premiumSum / grossPremiumSum if grossPremiumSum != 0 else 0
        combinedRatio = lossRatio + commissionRatio + netManagementExpenseRatio + netRetroExpenseRatio

        resultTableDictionary["Net Premium"]= np.append(resultTableDictionary["Net Premium"], premiumSum)
        resultTableDictionary["Net Incurred Claim"] = np.append(resultTableDictionary["Net Incurred Claim"], claimSum)
        resultTableDictionary["Net Commission"] = np.append(resultTableDictionary["Net Commission"], commissionSum)
        resultTableDictionary["Net Technical Margin"] = np.append(resultTableDictionary["Net Technical Margin"], netTechnicalMargin)
        resultTableDictionary["Loss Ratio"] = np.append(resultTableDictionary["Loss Ratio"],lossRatio)
        resultTableDictionary["Commission Ratio"] = np.append(resultTableDictionary["Commission Ratio"], commissionRatio)
        resultTableDictionary["Net Management Expense Ratio"] = np.append(resultTableDictionary["Net Management Expense Ratio"], netManagementExpenseRatioLocal)
        resultTableDictionary["Net Technical Margin Ratio"] = np.append(resultTableDictionary["Net Technical Margin Ratio"], netTechnicalMarginRatio)
        resultTableDictionary["Net Retro Expense Ratio"]= np.append(resultTableDictionary["Net Retro Expense Ratio"], netRetroExpenseRatio)
        resultTableDictionary["Combined Ratio"] = np.append(resultTableDictionary["Combined Ratio"], combinedRatio)
        grossPremiumSumArr = np.append(grossPremiumSumArr,grossPremiumSum)

ratioAnalysis()

resultTableDictionary["Net Premium"]= np.append(resultTableDictionary["Net Premium"], np.sum(resultTableDictionary["Net Premium"]))
resultTableDictionary["Net Incurred Claim"]= np.append(resultTableDictionary["Net Incurred Claim"], np.sum(resultTableDictionary["Net Incurred Claim"]))
resultTableDictionary["Net Commission"]= np.append(resultTableDictionary["Net Commission"], np.sum(resultTableDictionary["Net Commission"]))
resultTableDictionary["Net Technical Margin"]= np.append(resultTableDictionary["Net Technical Margin"], np.sum(resultTableDictionary["Net Technical Margin"]))

resultTableDictionary["Loss Ratio"] = np.append(resultTableDictionary["Loss Ratio"],
                                               abs((resultTableDictionary["Net Incurred Claim"])[-1]
                                                   /(resultTableDictionary["Net Premium"])[-1]) )
resultTableDictionary["Commission Ratio"] = np.append(resultTableDictionary["Commission Ratio"],
                                               abs((resultTableDictionary["Net Commission"])[-1]
                                                   /(resultTableDictionary["Net Premium"])[-1]) )
resultTableDictionary["Net Management Expense Ratio"] = np.append(resultTableDictionary["Net Management Expense Ratio"], netManagementExpenseRatio )
resultTableDictionary["Net Technical Margin Ratio"] = np.append(resultTableDictionary["Net Technical Margin Ratio"],
                                               abs((resultTableDictionary["Net Technical Margin"])[-1]
                                                   /(resultTableDictionary["Net Premium"])[-1]) )
resultTableDictionary["Net Retro Expense Ratio"] = np.append(resultTableDictionary["Net Retro Expense Ratio"],1 -
                                               abs((resultTableDictionary["Net Premium"])[-1]
                                                   /np.sum(grossPremiumSumArr)) )
resultTableDictionary["Combined Ratio"] = np.append(resultTableDictionary["Combined Ratio"],
                                                     resultTableDictionary["Loss Ratio"][-1] + resultTableDictionary["Commission Ratio"][-1] +\
                                                        resultTableDictionary["Net Management Expense Ratio"][-1] + resultTableDictionary["Net Retro Expense Ratio"][-1])


resultTable = pd.DataFrame(resultTableDictionary).reset_index(drop=True)

ratio_columns = [
    "Loss Ratio",
    "Commission Ratio",
    "Net Management Expense Ratio",
    "Net Technical Margin Ratio",
    "Net Retro Expense Ratio",
    "Combined Ratio"
]

name_columns = [
    "Net Premium",
    "Net Incurred Claim",
    "Net Commission",
    "Net Technical Margin"
]

resultTable[name_columns] = resultTable[name_columns].map(
    lambda x: f"({abs(x):,.2f})" if x < 0 else f"{x:,.2f}"
)

resultTable[ratio_columns] = resultTable[ratio_columns].map(lambda x: f"{x:.2%}")
print(premiumTable[dataFieldHeader].sum)
premiumTable


KeyboardInterrupt: 

In [7]:
categoryHeader = "Reporting Unit1 Leaf Name"
codeHeader = "Entry Code"
dataFieldHeader = "Retained Amt Base"
grossDataFieldHeader = "Detail Amount (Base)"
netManagementExpenseRatio = 0.12

premiumCode = np.array([100,500])
claimCode = np.array([300,600,900])
commissionCode = np.array([200])


filterItems = pd.unique(sourceData[categoryHeader].dropna())
sourceData[codeHeader] = sourceData[codeHeader].astype(str).str.strip()
sourceData[codeHeader] = pd.to_numeric(sourceData[codeHeader], errors="coerce")



def premiumClaimCommissionTableConstruct():

    premiumTable = pd.DataFrame(columns=sourceData.columns)
    claimTable = pd.DataFrame(columns=sourceData.columns)
    commissionTable = pd.DataFrame(columns=sourceData.columns)
        

    for code in premiumCode:
        premiumFilteredTable = sourceData[(sourceData[codeHeader] >= code) & (sourceData[codeHeader] <= (code + 99))]
        premiumTable = pd.concat([premiumTable, premiumFilteredTable], ignore_index=True)

    for code in claimCode:
        claimFilteredTable = sourceData[(sourceData[codeHeader] >= code) & (sourceData[codeHeader] <= (code + 99))]
        claimTable = pd.concat([claimTable,claimFilteredTable], ignore_index=True)

    for code in commissionCode:
        commissionFilteredTable = sourceData[(sourceData[codeHeader] >= code) & (sourceData[codeHeader] <= (code + 99))]
        commissionTable = pd.concat([commissionTable, commissionFilteredTable], ignore_index=True)

    return({
        "premiumTable": premiumTable,
        "claimTable": claimTable,
        "commissionTable": commissionTable
    })

tableDict = premiumClaimCommissionTableConstruct()
tableDict



{'premiumTable':       Business ID                                     Business Title  \
 0        NPF1003A                   CHEVRON NIGERIA LTD/ESCRAVOS GTL   
 1        NPF1003A                   CHEVRON NIGERIA LTD/ESCRAVOS GTL   
 2        NPF1004A         TOTAL UPSTREAM NIGERIA LIMITED (AKPO FPSO)   
 3        NPF1012A  INDORAMA ELEME PETROCHEMICALS/FERTILIZERS LIMITED   
 4        NPF1016A             TOTAL UPSTREAM NIGERIA LIMITED (EGINA)   
 ...           ...                                                ...   
 85648    PPT6376A                               ZAYTOUNA/ GEN ACC SP   
 85649    PPT6377A                              ZAYTOUNA/FIRE QS & SP   
 85650    PPT6378A                              ZAYTOUNA/ ENG QS & SP   
 85651    PPT6378A                              ZAYTOUNA/ ENG QS & SP   
 85652    PPT6389A                       ARC/SERENITY SA/BOND SURPLUS   
 
            Type of Business Underwriting Year  \
 0      Non-Prop Facultative              2023   
 1    

In [33]:
def ratioAnalysis(tableDict : dict, filterArray = None):
    
    resultTableDictionary = {
    categoryHeader:np.append(np.array(filterItems), "Total"),
    "Net Premium": np.array([]),
    "Net Incurred Claim": np.array([]),
    "Net Commission": np.array([]),
    "Net Technical Margin": np.array([]),
    "Loss Ratio": np.array([]),
    "Commission Ratio": np.array([]),
    "Net Management Expense Ratio": np.array([]),
    "Net Technical Margin Ratio": np.array([]),
    "Net Retro Expense Ratio": np.array([]),
    "Combined Ratio": np.array([])
    }

    grossPremiumSum = 0
    grossPremiumSumArr = np.array([])

    premiumTable = tableDict["premiumTable"]
    claimTable =  tableDict["claimTable"]
    commissionTable =  tableDict["commissionTable"]

    def modifyTable(filterLabel,filterItem, operation = "Equal"):
         
        nonlocal premiumTable, claimTable, commissionTable
        if operation == "Equal":
            premiumTable = premiumTable[premiumTable[filterLabel] == filterItem]
            claimTable = claimTable[claimTable[filterLabel] == filterItem]
            commissionTable = commissionTable[commissionTable[filterLabel] == filterItem]
        elif operation == "Atlest":
            premiumTable = premiumTable[premiumTable[filterLabel] >= filterItem]
            claimTable = claimTable[claimTable[filterLabel] >= filterItem]
            commissionTable = commissionTable[commissionTable[filterLabel]>= filterItem]
        elif operation == "Atmost":
            premiumTable = premiumTable[premiumTable[filterLabel] <= filterItem]
            claimTable = claimTable[claimTable[filterLabel] <= filterItem]
            commissionTable = commissionTable[commissionTable[filterLabel]<= filterItem]

    for item in filterItems:
        modifyTable(categoryHeader,item)

        if filterArray and len(filterArray) > 0:
            for filter_spec in filterArray:
                header = filter_spec[0]
                filterValue = filter_spec[1]
                operation = filter_spec[2]
                modifyTable(header, filterValue, operation)
                # if operation == "Equal":
                #     modifyTable(header, filterValue)
                # elif operation == "Atlest":
                #     modifyTable(header, filterValue, "Atlest")
                # elif operation == "Atmost":
                #     modifyTable(header, filterValue, "Atmost")

        premiumSum = np.sum(premiumTable[dataFieldHeader])
        grossPremiumSum = np.sum(premiumTable[grossDataFieldHeader])
        print(grossPremiumSum)
        claimSum = np.sum(claimTable[dataFieldHeader])
        commissionSum = np.sum(commissionTable[dataFieldHeader])

        netTechnicalMargin = premiumSum + claimSum + commissionSum
        lossRatio = abs(claimSum / premiumSum) if premiumSum != 0 else 0
        commissionRatio = abs(commissionSum / premiumSum) if premiumSum != 0 else 0
        netManagementExpenseRatioLocal = netManagementExpenseRatio
        netTechnicalMarginRatio = abs(netTechnicalMargin/premiumSum) if premiumSum != 0 else 0
        netRetroExpenseRatio = 1 - (premiumSum / grossPremiumSum) if grossPremiumSum != 0 else 0
        combinedRatio = lossRatio + commissionRatio + netManagementExpenseRatio + netRetroExpenseRatio

        resultTableDictionary["Net Premium"]= np.append(resultTableDictionary["Net Premium"], premiumSum)
        resultTableDictionary["Net Incurred Claim"] = np.append(resultTableDictionary["Net Incurred Claim"], claimSum)
        resultTableDictionary["Net Commission"] = np.append(resultTableDictionary["Net Commission"], commissionSum)
        resultTableDictionary["Net Technical Margin"] = np.append(resultTableDictionary["Net Technical Margin"], netTechnicalMargin)
        resultTableDictionary["Loss Ratio"] = np.append(resultTableDictionary["Loss Ratio"],lossRatio)
        resultTableDictionary["Commission Ratio"] = np.append(resultTableDictionary["Commission Ratio"], commissionRatio)
        resultTableDictionary["Net Management Expense Ratio"] = np.append(resultTableDictionary["Net Management Expense Ratio"], netManagementExpenseRatioLocal)
        resultTableDictionary["Net Technical Margin Ratio"] = np.append(resultTableDictionary["Net Technical Margin Ratio"], netTechnicalMarginRatio)
        resultTableDictionary["Net Retro Expense Ratio"]= np.append(resultTableDictionary["Net Retro Expense Ratio"], netRetroExpenseRatio)
        resultTableDictionary["Combined Ratio"] = np.append(resultTableDictionary["Combined Ratio"], combinedRatio)
        grossPremiumSumArr = np.append(grossPremiumSumArr,grossPremiumSum)

        premiumTable = tableDict["premiumTable"]
        claimTable =  tableDict["claimTable"]
        commissionTable =  tableDict["commissionTable"]
    
    resultTableDictionary["Net Premium"]= np.append(resultTableDictionary["Net Premium"], np.sum(resultTableDictionary["Net Premium"]))
    resultTableDictionary["Net Incurred Claim"]= np.append(resultTableDictionary["Net Incurred Claim"], np.sum(resultTableDictionary["Net Incurred Claim"]))
    resultTableDictionary["Net Commission"]= np.append(resultTableDictionary["Net Commission"], np.sum(resultTableDictionary["Net Commission"]))
    resultTableDictionary["Net Technical Margin"]= np.append(resultTableDictionary["Net Technical Margin"], np.sum(resultTableDictionary["Net Technical Margin"]))

    resultTableDictionary["Loss Ratio"] = np.append(resultTableDictionary["Loss Ratio"],
                                                abs((resultTableDictionary["Net Incurred Claim"])[-1]
                                                    /(resultTableDictionary["Net Premium"])[-1]) )
    resultTableDictionary["Commission Ratio"] = np.append(resultTableDictionary["Commission Ratio"],
                                                abs((resultTableDictionary["Net Commission"])[-1]
                                                    /(resultTableDictionary["Net Premium"])[-1]) )
    resultTableDictionary["Net Management Expense Ratio"] = np.append(resultTableDictionary["Net Management Expense Ratio"], netManagementExpenseRatio )
    resultTableDictionary["Net Technical Margin Ratio"] = np.append(resultTableDictionary["Net Technical Margin Ratio"],
                                                abs((resultTableDictionary["Net Technical Margin"])[-1]
                                                    /(resultTableDictionary["Net Premium"])[-1]) )
    resultTableDictionary["Net Retro Expense Ratio"] = np.append(resultTableDictionary["Net Retro Expense Ratio"],1 -
                                                abs((resultTableDictionary["Net Premium"])[-1]
                                                    /np.sum(grossPremiumSumArr)) )
    resultTableDictionary["Combined Ratio"] = np.append(resultTableDictionary["Combined Ratio"],
                                                        resultTableDictionary["Loss Ratio"][-1] + resultTableDictionary["Commission Ratio"][-1] +\
                                                            resultTableDictionary["Net Management Expense Ratio"][-1] + resultTableDictionary["Net Retro Expense Ratio"][-1])
    
    resultTable = pd.DataFrame(resultTableDictionary).reset_index(drop=True)

    # ratio_columns = [
    # "Loss Ratio",
    # "Commission Ratio",
    # "Net Management Expense Ratio",
    # "Net Technical Margin Ratio",
    # "Net Retro Expense Ratio",
    # "Combined Ratio"
    # ]

    # name_columns = [
    #     "Net Premium",
    #     "Net Incurred Claim",
    #     "Net Commission",
    #     "Net Technical Margin"
    # ]

    # resultTable[name_columns] = resultTable[name_columns].map(
    #     lambda x: f"({abs(x):,.2f})" if x < 0 else f"{x:,.2f}"
    # )

    # resultTable[ratio_columns] = resultTable[ratio_columns].map(lambda x: f"{x:.2%}")
    return resultTable

filterArr = [['Main Class of Business', 'Accident', 'Equal']]
resultTable = ratioAnalysis(tableDict,filterArr)
resultTable


2906243.894966
2335775.496796994
20399294.425018974
20636886.87832202
291868.913832
317471.4947509999
327704.756025
6823580.468460005
600000.0
552565.302816
153673.154319
159096.56
2885.726612
4637234.573103998


,Reporting Unit1 Leaf Name,Net Premium,Net Incurred Claim,Net Commission,Net Technical Margin,Loss Ratio,Commission Ratio,Net Management Expense Ratio,Net Technical Margin Ratio,Net Retro Expense Ratio,Combined Ratio
0,Nigeria Office,2813108.69,-826964.26,-887586.22,1098558.20,0.29,0.32,0.12,0.39,0.03,0.76
1,Tunisia Office,2334700.30,-302740.05,-566495.89,1465464.36,0.13,0.24,0.12,0.63,0.00,0.49
2,ZIMBABWE SUBSIDIARY,20354157.83,-5809196.50,-6909627.95,7635333.38,0.29,0.34,0.12,0.38,0.00,0.75
3,KENYA SUBSIDIARY,19463619.73,-10945618.01,-5940067.50,2577934.22,0.56,0.31,0.12,0.13,0.06,1.04
4,Africa,291868.91,-318694.37,-80425.42,-107250.87,1.09,0.28,0.12,0.37,0.00,1.49
5,Asia,317471.49,-7649.50,-89740.68,220081.32,0.02,0.28,0.12,0.69,0.00,0.43
6,MiddleEast,327704.76,-63425.20,-94461.52,169818.04,0.19,0.29,0.12,0.52,0.00,0.60
7,Ghana Office,4759624.25,-541045.87,-1646480.73,2572097.65,0.11,0.35,0.12,0.54,0.30,0.88
8,Latin America,600000.00,0.00,-165000.00,435000.00,0.00,0.28,0.12,0.72,0.00,0.40
9,Sierra Leone Office,552565.30,-6983.46,-185802.54,359779.30,0.01,0.34,0.12,0.65,0.00,0.47


In [40]:

previousTableLength = 0
wb = xl.Workbook("try.xlsx")
ws = wb.add_worksheet("table")

def addTable(ws,oneSheet = True):
    global previousTableLength

    verticalDis = 3
    tableList = resultTable.values.tolist()
    columns = [{"header": col} for col in resultTable.columns.tolist()]

    num_rows = len(resultTable)
    num_cols = len(resultTable.columns)

    start_row = previousTableLength + verticalDis 
    start_col = 1

    end_row = start_row + num_rows
    end_col = start_col + num_cols - 1

    start_col_letter = xl_col_to_name(start_col)
    end_col_letter = xl_col_to_name(end_col)

    
    table_range = f"{start_col_letter}{start_row}:{end_col_letter}{end_row}"
    if oneSheet:
        previousTableLength += end_row

    ws.write(f"{start_col_letter}{start_row-1}", "Hello")
    ws.add_table(table_range, {
        "data": tableList,
        "columns": columns
    })

for i in [1,2]:
    addTable(ws)

wb.close()


In [ ]:

def excelSheetsGenerator(analysisType,
                         categoryBundle,
                         oneSheet = True):

    output = io.BytesIO()
    workBook = xl.Workbook(output, {'in_memory': True})


    previousTableLength = 0
    def addTable(ws, table, category = None):

        verticalDis = 2
        
        tableList = table.values.tolist()
        columns = [{"header": col} for col in table.columns.tolist()]

        num_rows = len(table)
        num_cols = len(table.columns)

        start_row = previousTableLength + verticalDis 
        start_col = 1

        end_row = start_row + num_rows
        end_col = start_col + num_cols - 1

        start_col_letter = xl_col_to_name(start_col)
        end_col_letter = xl_col_to_name(end_col)

        
        table_range = f"{start_col_letter}{start_row}:{end_col_letter}{end_row}"
        if oneSheet:
            ws.write(f"{start_col_letter}{start_row-1}", str(category))
            previousTableLength += end_row

        ws.add_table(table_range, {
            "data": tableList,
            "columns": columns
        })


    if oneSheet:
        workSheet = workBook.add_worksheet(analysisFunParem["analysis type"])
        for category in categoryBundle:
            resultTable =  performanceAnalysis()
            addTable(workSheet, resultTable, category)

            workBook.close()
            output.seek(0)
    else:
        for category in categoryBundle:
            workSheet = workBook.add_worksheet(str(category)[0:min(30,len(category))])
            resultTable =  performanceAnalysis()
            addTable(workSheet, resultTable)

            workBook.close()
            output.seek(0)
    
    

IndentationError: expected an indented block after 'for' statement on line 12 (378029798.py, line 14)

In [42]:
b = "sekou"
b[0:min(2,5)]

'se'

In [3]:
import xlsxwriter as xl

wb = xl.Workbook("try.xlsx")
ws = wb.add_worksheet("table")

ws.add_table("A1:B3", {
    "data": [
        ["Alice", 90],
        ["Bob", 85]
    ],
    "columns": [
        {"header": "Name"},
        {"header": "Score"}
    ]
})

wb.close()  # ← THIS is critical

In [26]:
resultTable.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['Reporting Unit1 Leaf Name', 'Net Premium', 'Net Incurred Claim',
       'Net Commission', 'Net Technical Margin', 'Loss Ratio',
       'Commission Ratio', 'Net Management Expense Ratio',
       'Net Technical Margin Ratio', 'Net Retro Expense Ratio',
       'Combined Ratio'],
      dtype='str')>

In [4]:
arr = range(3,7)
np.array(arr)

array([3, 4, 5, 6])